# Correction : nettoyer un dataset avant le modèle

Objectif : préparer un dataset de logements avant d'entraîner un modèle.

On corrige les valeurs manquantes, les types incorrects, les doublons et les valeurs impossibles.


## 1. Charger le dataset sale

In [ ]:
import pandas as pd

data = pd.DataFrame({
    "surface": ["70", "?", "85 m2", "70", "-10", "120", "95", "110 m2", None, "60"],
    "pieces": [3, 4, 3, 3, 2, None, 4, 5, 3, 2],
    "quartier": ["centre", "nord", "centre", "centre", "sud", "ouest", "nord", "ouest", "centre", "sud"],
    "prix": [350000, 420000, 390000, 350000, 180000, 610000, 480000, 590000, 410000, 260000]
})

data


## 2. Observer les problèmes

Avant de corriger, on inspecte le dataset.

Questions :

- Combien y a-t-il de lignes et de colonnes ?
- Quelles colonnes ont un type incorrect ?
- Y a-t-il des valeurs manquantes ?
- Y a-t-il des doublons ?

In [ ]:
data.shape

data.head()


In [ ]:
data.info()


In [ ]:
data.isna().sum()

data.duplicated().sum()


## 3. Créer une copie de travail

Bonne pratique métier : on garde une version brute et on travaille sur une copie.

In [ ]:
data_clean = data.copy()

data_clean.head()


## 4. Corriger la colonne surface

La colonne `surface` contient du texte : `?`, `85 m2`, `110 m2`.

Objectif : obtenir une colonne numérique.

In [ ]:
data_clean["surface"] = data_clean["surface"].replace("?", None)
data_clean["surface"] = data_clean["surface"].astype(str).str.replace(" m2", "", regex=False)
data_clean["surface"] = pd.to_numeric(data_clean["surface"], errors="coerce")

data_clean


## 5. Gérer les valeurs manquantes

Bonne pratique : on ne remplace pas au hasard.

Ici, on remplace les valeurs manquantes de `surface` et `pieces` par la médiane.

In [ ]:
data_clean["surface"] = data_clean["surface"].fillna(data_clean["surface"].median())

data_clean["pieces"] = data_clean["pieces"].fillna(data_clean["pieces"].median())

data_clean.isna().sum()


## 6. Supprimer les doublons et les valeurs impossibles

Une surface négative est une valeur impossible métier.

On supprime aussi les lignes dupliquées.

In [ ]:
data_clean = data_clean.drop_duplicates()

data_clean = data_clean[data_clean["surface"] > 0]

data_clean


## 7. Transformer la colonne quartier

La colonne `quartier` est textuelle.

Pour utiliser un modèle scikit-learn, on la transforme en colonnes numériques avec `get_dummies`.

In [ ]:
data_clean = pd.get_dummies(data_clean, columns=["quartier"], dtype=int)

data_clean


## 8. Préparer X et y

`X` contient les colonnes utilisées pour prédire.

`y` contient la valeur à prédire.

In [ ]:
X = data_clean.drop(columns=["prix"])

y = data_clean["prix"]

print("X :", X.shape)
print("y :", y.shape)


## Questions finales

1. Pourquoi faut-il garder une version brute du dataset ?
2. Pourquoi une surface négative est-elle un problème métier ?
3. Pourquoi ne faut-il pas remplacer une valeur manquante au hasard ?
4. Pourquoi transforme-t-on `quartier` en colonnes numériques ?
5. À quel moment peut-on créer `X` et `y` ?